# 00 — Control Plane & Ingestion Gate (`Agent 11` / `Agents 01, 10`)

- **Active Run ID**: `run_20261111_002`
- **Stage Owner**: Agent 11 — Orchestration & QA (`src/pipeline/orchestrator.py`)
- **Contributing Modules**: Agent 01 (`src/ingest/`, `src/registry/`), Agent 10 (`src/artifact_registry.py`, `src/freshness_checker.py`)

## Responsibilities
1. Verify immutable raw source SHA-256 (`data/raw/survey_raw.csv`).
2. Run zero-hardcoding static scan across `src/` and `notebooks/`.
3. Execute or verify deterministic survey ingestion (`answers_normalized.parquet`, `question_registry.json`) and `verify_gate_1`.
4. Inspect `runs/run_20261111_002/manifest.json` provenance and evaluate pipeline readiness states.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/HFG'), Path('/content')]
    for p in candidates:
        if (p / 'src').is_dir() and ((p / 'pyproject.toml').is_file() or (p / 'data').is_dir()):
            return p.resolve()
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None and Path('/content').is_dir():
    target = Path('/content/HFG')
    print(f'Repository not found locally; cloning into {target}...')
    subprocess.run(['git', 'clone', 'https://github.com/necantis/HFG.git', str(target)], check=True)
    REPO_ROOT = _find_repo_root()

if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate HFG repo root from {Path.cwd()}.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'HFG repo root: {REPO_ROOT}')

from src.artifact_registry import ArtifactRegistry
from src.hardcoding_scanner import HardcodingScanner, verify_gate_1
from src.ingest.ingestion import ACTIVE_RUN_ID, EXPECTED_RAW_SHA256, ingest_survey
from src.registry.question_registry import build_question_registry

scanner = HardcodingScanner(base_dir=REPO_ROOT)
scan_violations = scanner.scan_codebase(['src', 'notebooks'])
assert len(scan_violations) == 0, f'Hardcoding violations detected: {scan_violations}'

raw_csv = REPO_ROOT / 'data' / 'raw' / 'survey_raw.csv'
raw_sha = ArtifactRegistry.compute_sha256(raw_csv)
assert raw_sha == EXPECTED_RAW_SHA256, f'Raw SHA mismatch: {raw_sha} != {EXPECTED_RAW_SHA256}'
print(f'Hardcoding scan: PASS (0 violations) | Raw SHA-256 verified: {raw_sha[:16]}...')


In [ ]:
import json

norm_parquet = REPO_ROOT / 'data' / 'normalized' / 'answers_normalized.parquet'
norm_json = REPO_ROOT / 'data' / 'normalized' / 'answers_normalized.json'
reg_json = REPO_ROOT / 'data' / 'normalized' / 'question_registry.json'

if not norm_parquet.is_file() or not reg_json.is_file():
    df_norm = ingest_survey(
        raw_csv_path=raw_csv,
        output_parquet=norm_parquet,
        output_json=norm_json,
        run_id=ACTIVE_RUN_ID,
    )
    build_question_registry(
        answers_df_or_path=df_norm,
        raw_csv_path=raw_csv,
        output_json=reg_json,
        run_id=ACTIVE_RUN_ID,
    )
    print(f'Ingested normalized survey rows: {len(df_norm)}')

gate1 = verify_gate_1(base_dir=REPO_ROOT, run_id=ACTIVE_RUN_ID)
assert gate1['status'] == 'PASS', f'Gate 1 failed: {gate1}'
print('Agent 01 Gate 1:', json.dumps(gate1, indent=2))

registry = ArtifactRegistry(base_dir=REPO_ROOT, run_id=ACTIVE_RUN_ID)
manifest_path = REPO_ROOT / 'runs' / ACTIVE_RUN_ID / 'manifest.json'
if manifest_path.is_file():
    registry.load_manifest(manifest_path)
    print(f'Loaded manifest with {len(registry.artifacts)} artifact(s): {list(registry.artifacts.keys())}')


In [ ]:
from src.pipeline.orchestrator import evaluate_pipeline_state

write_log = os.environ.get('HFG_WRITE_EXECUTION_LOG') == '1'
report = evaluate_pipeline_state(base_dir=REPO_ROOT, run_id=ACTIVE_RUN_ID, write_log=write_log)
print('Highest verified state:', report.highest_verified_state)
print('States:', json.dumps(report.states, indent=2))
print(f'AGENT_03_RELEASE = {report.agent_03_release_decision.value}')
